<a href="https://colab.research.google.com/github/muhammadrizki2001/DataScience_240401010030_Muhammad-Rizki/blob/main/Pertemuan12_Muhammad_Rizki_240401010030.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Generate & Eksplorasi Dataset Transaksi

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Generate 50 transaksi sintetis dengan jumlah item 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
    list(map(str, np.random.choice(produk, n_item, replace=False)))
)

# Menambahkan pola pembelian bersama antara Roti dan Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

In [ ]:
# One-Hot Encoding Transaksi

from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

In [ ]:
# Cari Frequent Itemset dengan Apriori

from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Memilih nilai minimum support yang menghasilkan frequent itemset dengan jumlah yang representatif
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

In [ ]:
# Bentuk & Saring Aturan Asosiasi

from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence',
                           min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))

# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

Berdasarkan hasil association rule, aturan dengan nilai lift tertinggi menunjukkan hubungan pembelian yang paling kuat. Salah satu aturan yang relevan secara bisnis adalah Roti → Selai, karena pelanggan yang membeli Roti memiliki kecenderungan lebih tinggi untuk membeli Selai dibandingkan jika pembelian tersebut terjadi secara acak.

In [ ]:
# Rekomender Sederhana dengan Content-Based Filtering

from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

In [ ]:
# Bandingkan Kedua Pendekatan

produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

## Perbandingan Pendekatan Rekomendasi

Hasil rekomendasi dari Association Rule dan Content-Based Filtering menunjukkan hasil yang cukup konsisten, yaitu keduanya merekomendasikan Selai sebagai produk yang berkaitan dengan Roti.

Association Rule memberikan rekomendasi berdasarkan pola pembelian pelanggan, sedangkan Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan kategori produk.

Dalam praktiknya, kedua pendekatan dapat digabungkan menjadi sistem hybrid agar rekomendasi memanfaatkan pola perilaku pengguna sekaligus karakteristik produk.